In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(DATA_PATH / "train.csv")

print("Original shape:", train_df.shape)
print(train_df.head())

Original shape: (1034, 11)
  record_id        date  students_on_campus  menu_price_rs  temp_c  \
0    R00001  2026-01-01                5151           50.0    25.4   
1    R00002  2026-01-01                5280           47.0    44.1   
2    R00002  2026-01-01                5280           47.0    44.1   
3    R00003  2026-01-01                5328           82.0    42.6   
4    R00004  2026-01-02                1208           54.0    43.1   

   exam_days_left  special_dishes  queue_minutes_yesterday  day_type    block  \
0               9               1                     32.6   weekday   Hostel   
1              51               2                     31.3  saturday   Hostel   
2              51               2                     31.3  saturday   Hostel   
3              30               5                     12.1   weekday   Hostel   
4              34               2                     26.4   weekday  A-Block   

   plates_sold  
0          441  
1          397  
2          397

In [2]:
# Keep original data unchanged
clean_df = train_df.copy()

# 1. Remove exact duplicate rows
clean_df = clean_df.drop_duplicates()

# 2. Correct menu price unit mix-up in B-Block
mask = clean_df["block"] == "B-Block"
clean_df.loc[mask, "menu_price_rs"] = (
    clean_df.loc[mask, "menu_price_rs"] / 100
)

# 3. Correct target values typed 10 times too large
mask = clean_df["plates_sold"] >= 1000
clean_df.loc[mask, "plates_sold"] = (
    clean_df.loc[mask, "plates_sold"] / 10
)

print("Original rows:", len(train_df))
print("Cleaned rows:", len(clean_df))
print("Duplicates removed:", len(train_df) - len(clean_df))
print("\nCorrected B-Block prices:")
print(clean_df.loc[clean_df["block"] == "B-Block", "menu_price_rs"].describe())
print("\nCorrected target maximum:", clean_df["plates_sold"].max())

Original rows: 1034
Cleaned rows: 1000
Duplicates removed: 34

Corrected B-Block prices:
count    234.000000
mean      75.051282
std       26.811005
min       30.000000
25%       53.000000
50%       77.000000
75%       99.000000
max      119.000000
Name: menu_price_rs, dtype: float64

Corrected target maximum: 655


In [3]:
clean_df.to_csv(OUTPUT_PATH / "train_clean.csv", index=False)

print("Saved:", OUTPUT_PATH / "train_clean.csv")

Saved: d:\College_Canteen_lunchRush\College_Canteen_lunchRush\data\processed\train_clean.csv


In [4]:
assert clean_df.duplicated().sum() == 0
assert clean_df["menu_price_rs"].max() < 200
assert clean_df["plates_sold"].max() < 1000

print("All cleaning checks passed!")

All cleaning checks passed!
